# Holdout Eval: Base vs SFT vs DPO vs KTO

Runs greedy generation on the 100 held-out CharXiv reasoning questions for the base model and three LoRA adapters. Adapters are loaded from completed training kernel outputs.

In [ ]:
import subprocess
import sys

# Pin a known-good stack. Current Kaggle images ship transformers 5.x + torch 2.10,
# which break PEFT imports via flex_attention (TransformGetItemToIndex).
# torch 2.5.1+cu124 also keeps P100 (sm_60) working when Kaggle assigns one instead of T4.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'transformers', 'peft', 'accelerate'],
    check=False,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.5.1', 'torchvision==0.20.1',
        '--index-url', 'https://download.pytorch.org/whl/cu124',
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1',
        'qwen-vl-utils==0.0.14', 'pillow',
    ],
    check=True,
)

import torch
import transformers
import peft

print(f'Active PyTorch: {torch.__version__}')
print(f'transformers: {transformers.__version__}, peft: {peft.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required for holdout evaluation.')
props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
print(f'GPU: {props.name}, cc={cc}, VRAM={props.total_memory / 1e9:.1f} GB')
if cc[0] < 6:
    raise RuntimeError(f'GPU compute capability {cc} is too old.')
print('Packages successfully configured.')

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

repo_dir = Path('/tmp/prm_project')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)
subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/prm_project.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir / 'src'))

from chart_prm.generator import build_generation_prompt

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SPLIT_PATH = Path('data/splits/eval_reasoning_ids.json')
QUESTIONS_PATH = Path('data/CharXiv/data/reasoning_val.json')
IMAGES_DIR = Path('data/CharXiv/images')
OUTPUT_PATH = Path('/kaggle/working/holdout_generations.jsonl')
SUMMARY_PATH = Path('/kaggle/working/holdout_accuracy.json')

ADAPTER_CANDIDATES = {
    'sft': [
        Path('/kaggle/input/qwen-vl-sft-custom/qwen_vl_sft_adapter'),
        Path('/kaggle/input/qwen-vl-sft-custom'),
    ],
    'dpo': [
        Path('/kaggle/input/qwen-vl-step-dpo-custom/qwen_vl_dpo_adapter'),
        Path('/kaggle/input/qwen-vl-step-dpo-custom'),
    ],
    'kto': [
        Path('/kaggle/input/qwen-vl-kto-custom/qwen_vl_kto_adapter'),
        Path('/kaggle/input/qwen-vl-kto-custom'),
    ],
}


def resolve_adapter(name: str) -> Path:
    for candidate in ADAPTER_CANDIDATES[name]:
        if (candidate / 'adapter_config.json').exists():
            return candidate
        matches = list(candidate.rglob('adapter_config.json')) if candidate.exists() else []
        if matches:
            return matches[0].parent
    raise FileNotFoundError(f'Could not resolve {name} adapter under {ADAPTER_CANDIDATES[name]}')


adapter_paths = {name: resolve_adapter(name) for name in ADAPTER_CANDIDATES}
for name, path in adapter_paths.items():
    print(f'{name}: {path}')

if not torch.cuda.is_available():
    raise RuntimeError('Evaluation requires a CUDA GPU.')

with SPLIT_PATH.open(encoding='utf-8') as handle:
    eval_ids = [str(value) for value in json.load(handle)]
with QUESTIONS_PATH.open(encoding='utf-8') as handle:
    question_data = json.load(handle)

missing_image_ids = [qid for qid in eval_ids if not list(IMAGES_DIR.glob(f'{qid}.*'))]
if missing_image_ids:
    print(f'Downloading {len(missing_image_ids)} missing holdout images...')
    subprocess.run([sys.executable, 'scripts/download_images.py', '--ids-file', str(SPLIT_PATH)], check=True)

print(f'Holdout size: {len(eval_ids)}')

In [ ]:
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = PeftModel.from_pretrained(base_model, str(adapter_paths['sft']), adapter_name='sft')
model.load_adapter(str(adapter_paths['dpo']), adapter_name='dpo')
model.load_adapter(str(adapter_paths['kto']), adapter_name='kto')
model.eval()
print('Loaded base model with sft/dpo/kto adapters.')

In [ ]:
FINAL_ANSWER_RE = re.compile(r'Final Answer:\s*(.+)', re.IGNORECASE)


def extract_final_answer(text: str) -> str:
    matches = FINAL_ANSWER_RE.findall(text or '')
    if not matches:
        return ''
    return matches[-1].strip().strip('"\'`')


def normalize_answer(text: str) -> str:
    return re.sub(r'\s+', ' ', (text or '').strip().lower())


def generate_response(active_model, image_path: Path, question: str) -> str:
    prompt = build_generation_prompt(question)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to('cuda')
    with torch.inference_mode():
        generated_ids = active_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


def load_done_ids(path: Path) -> set:
    done = set()
    if not path.exists():
        return done
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            done.add(json.loads(line)['question_id'])
    return done


done_ids = load_done_ids(OUTPUT_PATH)
print(f'Resuming with {len(done_ids)} completed questions.')

systems = ['base', 'sft', 'dpo', 'kto']
with OUTPUT_PATH.open('a', encoding='utf-8') as handle:
    for idx, question_id in enumerate(eval_ids):
        if question_id in done_ids:
            continue
        record = question_data.get(question_id)
        if record is None:
            raise KeyError(f'Held-out question {question_id} missing from reasoning_val.json')
        image_matches = sorted(IMAGES_DIR.glob(f'{question_id}.*'))
        if not image_matches:
            raise FileNotFoundError(f'No image found for held-out question {question_id}')

        responses = {}
        with model.disable_adapter():
            responses['base'] = generate_response(model, image_matches[0], record['query'])
        for adapter_name in ('sft', 'dpo', 'kto'):
            model.set_adapter(adapter_name)
            responses[adapter_name] = generate_response(model, image_matches[0], record['query'])

        row = {
            'question_id': question_id,
            'question': record['query'],
            'ground_truth': record['answer'],
            'responses': responses,
            'predicted_answers': {name: extract_final_answer(text) for name, text in responses.items()},
        }
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()
        done_ids.add(question_id)
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f'[{idx + 1}/{len(eval_ids)}] completed question_id={question_id}')

print(f'Saved generations to {OUTPUT_PATH}')

In [ ]:
rows = []
with OUTPUT_PATH.open(encoding='utf-8') as handle:
    for line in handle:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

if len(rows) != len(eval_ids):
    raise AssertionError(f'Expected {len(eval_ids)} rows, found {len(rows)}')

summary = {'n': len(rows), 'exact_match': {}, 'extracted_answer_rate': {}}
for system in systems:
    correct = 0
    extracted = 0
    for row in rows:
        pred = row['predicted_answers'].get(system, '')
        if pred:
            extracted += 1
        if normalize_answer(pred) == normalize_answer(row['ground_truth']):
            correct += 1
    summary['exact_match'][system] = {
        'correct': correct,
        'accuracy': correct / len(rows),
    }
    summary['extracted_answer_rate'][system] = extracted / len(rows)

SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print(f'Saved summary to {SUMMARY_PATH}')